# Beta-Binomial Posterior Update

We have a coin with unknown bias θ. We'll flip it 200 times and, at each step,
update our belief about θ using Bayes' rule.

**The setup:** our belief is a Beta distribution. One head shifts it right;
one tail shifts it left. After enough flips, the data overwhelms the prior.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from scipy.stats import beta as beta_dist

In [ ]:
# Style
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('seaborn-whitegrid')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

# Color palette
C_PRIOR     = '#4C72B0'
C_POST      = '#DD8452'
C_TRUE      = '#55A868'
TRUE_THETA  = 0.7

# Shared theta grid (avoids singularity at 0 and 1)
THETA = np.linspace(0.001, 0.999, 500)

# Shared dataset — seeded for reproducibility; Act 3 "New dataset" button resamples
rng  = np.random.default_rng(42)
data = rng.binomial(1, TRUE_THETA, 200)

## Act 1: Your Prior Belief

Before seeing any data, we commit to a starting belief about θ.

A **Beta(α, β)** distribution encodes this:
- **α** acts like "prior heads" — higher α pulls the curve toward 1
- **β** acts like "prior tails" — higher β pulls the curve toward 0
- **Effective sample size = α + β** — controls how confident you are

Drag the sliders to see how the shape changes.

In [ ]:
# --- Shared widgets (reused in Act 3) ---
alpha_slider = widgets.FloatSlider(
    value=3.0, min=0.5, max=15.0, step=0.5,
    description='α  (prior heads)',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='380px')
)
beta_slider = widgets.FloatSlider(
    value=2.0, min=0.5, max=15.0, step=0.5,
    description='β  (prior tails)',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='380px')
)

prior_out = widgets.Output()

def _draw_prior(_=None):
    a, b = alpha_slider.value, beta_slider.value
    pdf  = beta_dist.pdf(THETA, a, b)
    mean = a / (a + b)
    ess  = a + b
    with prior_out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(THETA, pdf, color=C_PRIOR, linewidth=2.5)
        ax.fill_between(THETA, pdf, alpha=0.2, color=C_PRIOR)
        ax.axvline(mean, color=C_PRIOR, linestyle='--', linewidth=1.5, alpha=0.8,
                   label=f'Prior mean = {mean:.2f}')
        ax.set_xlabel('Probability θ could be the true coin bias')
        ax.set_ylabel('Belief density')
        ax.set_title(f'Prior  ·  Mean: {mean:.2f}  ·  Effective sample size: {ess:.0f}',
                     fontsize=12)
        ax.set_xlim(0, 1)
        ax.legend(fontsize=10)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.tight_layout()
        plt.show()

alpha_slider.observe(_draw_prior, names='value')
beta_slider.observe(_draw_prior, names='value')

display(widgets.VBox([
    widgets.HTML('<b>Shape your prior belief about the coin bias:</b>'),
    widgets.HBox([alpha_slider, beta_slider]),
    prior_out,
]))
_draw_prior()